In [2]:
%matplotlib qt
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
import glob
from scipy.ndimage import gaussian_filter
from utils import *

In [3]:
folder = '/home/ulyanov/data/solo/phi/flat/fdt/flat_newprefilter/'
flat_folders = sorted(glob.glob(folder + '*'))
flat_folders

['/home/ulyanov/data/solo/phi/flat/fdt/flat_newprefilter/2023-04-03',
 '/home/ulyanov/data/solo/phi/flat/fdt/flat_newprefilter/2023-04-06',
 '/home/ulyanov/data/solo/phi/flat/fdt/flat_newprefilter/2023-04-15',
 '/home/ulyanov/data/solo/phi/flat/fdt/flat_newprefilter/2023-10-11',
 '/home/ulyanov/data/solo/phi/flat/fdt/flat_newprefilter/2024-03-30',
 '/home/ulyanov/data/solo/phi/flat/fdt/flat_newprefilter/2024-09-26',
 '/home/ulyanov/data/solo/phi/flat/fdt/flat_newprefilter/2024-10-16',
 '/home/ulyanov/data/solo/phi/flat/fdt/flat_newprefilter/2024-10-27',
 '/home/ulyanov/data/solo/phi/flat/fdt/flat_newprefilter/2024-12-02',
 '/home/ulyanov/data/solo/phi/flat/fdt/flat_newprefilter/2025-01-19',
 '/home/ulyanov/data/solo/phi/flat/fdt/flat_newprefilter/2025-03-10',
 '/home/ulyanov/data/solo/phi/flat/fdt/flat_newprefilter/2025-09-15',
 '/home/ulyanov/data/solo/phi/flat/fdt/flat_newprefilter/2025-09-23',
 '/home/ulyanov/data/solo/phi/flat/fdt/flat_newprefilter/2026-03-10',
 '/home/ulyanov/data

In [7]:
flat_folder = flat_folders[-7]
flat_files = sorted(glob.glob(flat_folder + '/*.fits'))
flat_files

['/home/ulyanov/data/solo/phi/flat/fdt/flat_newprefilter/2024-12-02/phi-fdt-flat_20241202T123003_V202607281911C_0472020100.fits',
 '/home/ulyanov/data/solo/phi/flat/fdt/flat_newprefilter/2024-12-02/phi-fdt-ghost_20241202T123003_V202607281911C_0472020100.fits']

In [8]:
dark_file = '/home/ulyanov/data/solo/phi/dark/solo_CAL1_phi-fdt-dark_20240205T033810_V202402220119C_0422051001.fits.gz'
flat_file, ghost_file = flat_files

with fits.open(dark_file) as hdul:
    dark = hdul[0].data

with fits.open(flat_file) as hdul:
    flat = hdul[0].data

with fits.open(ghost_file) as hdul:
    ghost = hdul[0].data

ghost = demodulate(ghost)

In [10]:
plt.figure(figsize=(10,10))
plt.imshow(flat[0])

In [5]:
def get_wv_shift(data, cpos=5, pol=0, delta_wv=0.069):
    temp = data.copy().reshape((6, -1, data.shape[-2], data.shape[-1]))[:,pol]
    temp = np.delete(temp, cpos, axis=0)

    t = np.argmin(temp, axis=0)
    l, a, r = np.take_along_axis(temp, np.array([(t - 1) % 5, t, (t + 1) % 5]), axis=0)
    b, c = (r - l) / 2, (l + r) - 2 * a

    with np.errstate(invalid='ignore'):
        return (t - 2 - b / c) * delta_wv


def calc_ghost_scaling(data, cpos=5, delta_wv=0.069, sigma=0.043, gamma=0.053, depth=0.66):
    from scipy.signal import convolve

    dx = 0.001
    x = np.arange(-10,10 + dx / 2, dx)
    f = 1 - depth * np.exp(-x ** 2 / 2 / sigma ** 2)
    g = 1 / (1 + (x / gamma) ** 2)
    q = 2 * (1 - convolve(f, g ** 2, mode='same') / convolve(f, g, mode='same'))

    wv = np.arange(-2,3) * delta_wv
    shift = get_wv_shift(data, cpos=cpos, delta_wv=delta_wv)

    Q = interpolate(q.reshape(-1,1,1), x, wv.reshape(-1,1,1) - np.expand_dims(shift, axis=0))
    return np.insert(np.nan_to_num(Q, nan=1), cpos, np.ones_like(shift), axis=0)

In [6]:
#folder = '/home/ulyanov/data/solo/phi/flat/fdt/calibration/2026-03-10/'
folder = '/home/ulyanov/data/solo/phi/test/'
files = sorted(glob.glob(folder + '*.fits.gz'))
files

['/home/ulyanov/data/solo/phi/test/solo_L1_phi-fdt-alam_20260805T010009_V202608050434C_0648050501.fits.gz',
 '/home/ulyanov/data/solo/phi/test/solo_L1_phi-fdt-ilam_20240101T040003_V202401090117C_0441010503.fits.gz',
 '/home/ulyanov/data/solo/phi/test/solo_L1_phi-fdt-ilam_20240106T210007_V202401100517C_0441060508.fits.gz',
 '/home/ulyanov/data/solo/phi/test/solo_L1_phi-fdt-ilam_20240107T000009_V202401110118C_0441070501.fits.gz',
 '/home/ulyanov/data/solo/phi/test/solo_L1_phi-fdt-ilam_20240207T000009_V202402130123C_0442070501.fits.gz',
 '/home/ulyanov/data/solo/phi/test/solo_L1_phi-fdt-ilam_20240207T060009_V202402130123C_0442070502.fits.gz',
 '/home/ulyanov/data/solo/phi/test/solo_L1_phi-fdt-ilam_20240318T190009_V202405151841C_0443180504.fits.gz',
 '/home/ulyanov/data/solo/phi/test/solo_L1_phi-fdt-ilam_20240328T060009_V202405152307C_0443281521.fits.gz',
 '/home/ulyanov/data/solo/phi/test/solo_L1_phi-fdt-ilam_20240330T044009_V202405152319C_0443301531.fits.gz',
 '/home/ulyanov/data/solo/ph

In [33]:
with fits.open(files[0]) as hdul:
    header = hdul[0].header
    data = hdul[0].data

xr, yr = reflection_point_predict(header)
cpos = header['CONTPOS'] - 1
print(cpos)

nx, ny = data.shape[-2:]

data = data.reshape(6,4,nx,ny)
data -= crop(dark, header) * 0.4
data /= crop(flat, header)

q = calc_ghost_scaling(data, cpos=cpos)

5


In [34]:
i = cpos

temp = data[i].copy()
temp = realign(temp)
temp = demodulate(temp)

reflection = reflect(gaussian_filter(temp[0], 8), xr, yr)

In [37]:
j = 3

plt.figure(figsize=(10,10))
plt.imshow(temp[j] - crop(ghost, header)[j] * reflection * q[i], vmin=-30, vmax=30)
plt.tight_layout()

In [18]:
j = 3

plt.figure(figsize=(10,10))
plt.imshow(temp[j], vmin=-30, vmax=30)
plt.tight_layout()